# 07 - Foundry IQ through Azure API Management

Notebook 06 called the knowledge base **directly**, so the notebook itself held the Web IQ key and sent the user's Search token. Here the same three sources (Blob, SharePoint, Web IQ) are reached **through an APIM gateway** instead.

```text
notebook --(Entra token for the broker app)--> APIM /iq
                                                 | validate-jwt
                                                 | on-behalf-of: same user -> Azure AI Search token
                                                 | add Web IQ key from an APIM secret
                                                 | call Search as the APIM managed identity
                                                 v
                                        knowledge base --> Blob + SharePoint + Web IQ
```

What changes for the caller:

| | Notebook 06 (direct) | Notebook 07 (APIM) |
|---|---|---|
| Caller's token | Azure AI Search | the **broker app** (a different app registration) |
| Web IQ key | in the notebook | only in APIM |
| SharePoint permissions | caller's Search token | same user, via the on-behalf-of exchange |
| Caller needs Search RBAC | yes | no, only the broker token |

You will create a Web IQ source and a knowledge base that reuse notebook 06's Blob and SharePoint sources, then ask questions only through the gateway.

## Prerequisites

- Notebook 06 completed with `CLEAN_UP = False`, so `{prefix}-kb-blob` and `{prefix}-kb-sharepoint` exist.
- The facilitator ran `scripts\deploy.ps1` (creates APIM), `scripts\setup-broker-app.ps1` (the broker app registration) and `scripts\setup-apim-policy.ps1` (the gateway policy), and gave you `BROKER_APP_CLIENT_ID`, `APIM_GATEWAY_URL` and `WEBIQ_APIM_SOURCE_NAME` for `.env`.
- You are signed in with `az login` as your own user.

In [ ]:
import base64
import json
import os
import re
from pathlib import Path
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'BROKER_APP_CLIENT_ID', 'APIM_GATEWAY_URL')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Ask the facilitator; see .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
broker_client_id = str(UUID(os.environ['BROKER_APP_CLIENT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
apim_url = os.environ['APIM_GATEWAY_URL'].rstrip('/')
api_version = '2026-08-01-preview'

blob_source_name = f'{prefix}-kb-blob'
sharepoint_source_name = f'{prefix}-kb-sharepoint'
webiq_source_name = os.getenv('WEBIQ_APIM_SOURCE_NAME', 'iq-workshop-webiq')
knowledge_base_name = f'{prefix}-apim-kb'

credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
broker_token = get_bearer_token_provider(credential, f'api://{broker_client_id}/user_impersonation')
print('Gateway:', apim_url)
print('Knowledge base:', knowledge_base_name)

## 1. Get a token for the broker app

The caller no longer asks for an Azure AI Search token. It asks for a token whose audience is the **broker app registration**, using its `user_impersonation` scope. The setup script pre-authorized Azure CLI for that scope, so there is no consent prompt.

The cell decodes the token (reading only; APIM does the real validation) so you can see who it is for.

In [ ]:
claims = json.loads(base64.urlsafe_b64decode(broker_token().split('.')[1] + '==='))
print('audience (aud):', claims['aud'])
print('scope (scp):   ', claims.get('scp'))
print('user:          ', claims.get('preferred_username') or claims.get('upn'))
if claims['aud'] not in (broker_client_id, f'api://{broker_client_id}'):
    raise RuntimeError('The token is not for the broker app.')

## 2. Create the Web IQ source and the knowledge base

The Blob and SharePoint sources already exist from notebook 06. The Web IQ source is created here as one **shared** source, named in the APIM policy (`WEBIQ_APIM_SOURCE_NAME`). Like notebook 06 it stores no key; APIM adds the key as per-source control headers on every request. Creating the same shared source again is harmless.

These are control-plane calls straight to Search with your own token, as in the earlier notebooks. Only *retrieval* goes through APIM.

In [ ]:
webiq_source = {
    'name': webiq_source_name, 'kind': 'mcpServer',
    'description': 'Web IQ web grounding for current public information.',
    'resultsProcessing': 'rerank',
    'mcpServerParameters': {
        'serverURL': 'https://api.microsoft.ai/v3/mcp',
        'authentication': None,
        'tools': [{'name': 'web', 'outputParsing': {'kind': 'auto'}, 'maxOutputTokens': 2000}],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{webiq_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=webiq_source, timeout=(10, 120),
)
response.raise_for_status()
print('Web IQ knowledge source ready:', webiq_source_name)

In [ ]:
source_kinds = {}
for source_name in (blob_source_name, sharepoint_source_name):
    response = requests.get(
        f'{search_endpoint}/knowledgesources/{source_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    if response.status_code == 404:
        raise RuntimeError(f'{source_name} not found. Complete notebook 06 first (CLEAN_UP = False).')
    response.raise_for_status()
    source_kinds[source_name] = response.json()['kind']
source_kinds[webiq_source_name] = 'mcpServer'
print('Source kinds:', source_kinds)

# Reuse the chat model settings from notebook 06's knowledge base.
response = requests.get(
    f'{search_endpoint}/knowledgebases/{prefix}-kb?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
if response.status_code == 404:
    raise RuntimeError(f'{prefix}-kb not found. Complete notebook 06 first (CLEAN_UP = False).')
response.raise_for_status()
base_kb = response.json()

knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Workshop policies (Blob and SharePoint) plus Web IQ, reached through APIM.',
    'retrievalInstructions': (
        f'Use {blob_source_name} and {sharepoint_source_name} for questions about internal company policy, '
        f'travel, onboarding or documents. Use {webiq_source_name} only for current public information.'
    ),
    'answerInstructions': base_kb.get('answerInstructions'),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [{'name': name} for name in source_kinds],
    'models': base_kb['models'],
    'retrievalReasoningEffort': {'kind': 'low'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()
print('Knowledge base ready:', knowledge_base_name)

## 3. Ask through the gateway

Look at the request headers: only the broker token. There is no Web IQ key, no `x-ms-query-source-authorization` and no Search token. APIM supplies all of them. The URL is the APIM gateway (`/iq/...`), not the Search endpoint.

Start with an internal question.

In [ ]:
question = 'What is our policy for booking international travel?'
response = requests.post(
    f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {broker_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': source_kinds[blob_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': source_kinds[sharepoint_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

The next question needs current public information. The Web IQ key is added by APIM, so the notebook never touches it.

In [ ]:
question = 'What is the latest stable release of Python, and when was it released?'
response = requests.post(
    f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {broker_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': source_kinds[blob_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': source_kinds[sharepoint_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

Finally a question that needs both: internal policy and the outside world.

In [ ]:
question = 'Does our travel policy meet current public guidance on business travel? Compare them.'
response = requests.post(
    f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {broker_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 90,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': source_kinds[blob_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': source_kinds[sharepoint_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 150),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

## 4. Check that the gateway rejects the wrong callers

APIM's `validate-jwt` policy accepts only tokens for the broker app with the `user_impersonation` scope. Two calls that should fail with **401**:

1. no token at all
2. a valid Azure AI Search token (right user, wrong audience)

In [ ]:
url = f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}'
body = {'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'hello'}]}]}

no_token = requests.post(url, json=body, timeout=(10, 60))
print('No token:', no_token.status_code)

wrong_audience = requests.post(
    url, headers={'Authorization': f'Bearer {search_token()}'}, json=body, timeout=(10, 60),
)
print('Search token (wrong audience):', wrong_audience.status_code)

if no_token.status_code != 401 or wrong_audience.status_code != 401:
    raise RuntimeError('The gateway should reject both calls with 401. Check the APIM policy.')

## Cleanup

Deletes only your knowledge base. The Web IQ source is **shared by every attendee**, so it is kept (delete it only after the workshop). Notebook 06's cleanup removes the Blob and SharePoint sources.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    response = requests.delete(
        f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    if response.status_code != 404:
        response.raise_for_status()
    print('Deleted or already absent:', knowledge_base_name)
else:
    print('Cleanup disabled.')

## References

- [Foundry IQ and agentic retrieval](https://learn.microsoft.com/azure/search/agentic-retrieval-overview)
- [Per-user access with `x-ms-query-source-authorization`](https://learn.microsoft.com/azure/search/search-document-level-access-overview)
- [Microsoft identity platform: on-behalf-of flow](https://learn.microsoft.com/entra/identity-platform/v2-oauth2-on-behalf-of-flow)
- [APIM `validate-jwt` policy](https://learn.microsoft.com/azure/api-management/validate-jwt-policy)
- [APIM `authentication-managed-identity` policy](https://learn.microsoft.com/azure/api-management/authentication-managed-identity-policy)